# Spherical CNN Architecture Visualization

Two methods to visualize how a Spherical CNN processes data:
1. **Block Diagram** - Layer flow and data shapes
2. **3D Spherical Visualization** - Convolution kernels and receptive fields on the sphere

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

## Method 1: Block Diagram (Graphviz)

Clean, publication-ready architecture diagram showing layer flow and shapes.

In [ ]:
# Create spherical CNN architecture diagram using matplotlib
# (No external graphviz dependency needed)

fig, ax = plt.subplots(figsize=(14, 10))

# Define layer boxes (text, x_col, y_row, color)
layers = [
    ("Input\nnside=256\nNPIX=1.5M\n3 channels", 0, 9, 'lightgreen'),
    ("SphericalConv2D\n32 filters (3×3)", 0, 8, 'lightyellow'),
    ("BatchNorm + ReLU", 0, 7, 'lightyellow'),
    ("Pool nside→128\nNPIX=393K", 0, 6, 'lightcyan'),
    ("SphericalConv2D\n64 filters (3×3)", 0, 5, 'lightyellow'),
    ("BatchNorm + ReLU", 0, 4, 'lightyellow'),
    ("Pool nside→64\nNPIX=98K", 0, 3, 'lightcyan'),
    ("SphericalConv2D\n128 filters (3×3)", 1, 9, 'lightyellow'),
    ("BatchNorm + ReLU", 1, 8, 'lightyellow'),
    ("Pool nside→32\nNPIX=24K", 1, 7, 'lightcyan'),
    ("Global Mean Pool", 1, 6, 'lightblue'),
    ("Dense 256", 1, 5, 'lightblue'),
    ("Output (10 classes)\nSoftmax", 1, 4, 'lightcoral'),
]

# Draw boxes and labels
for text, col, row, color in layers:
    x = col * 3 + 1
    y = row
    
    rect = plt.Rectangle((x - 0.8, y - 0.35), 1.6, 0.7, 
                         facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=9, weight='bold')

# Draw arrows connecting layers
connections = [
    ((1, 9), (1, 8)),      # Input → Conv1
    ((1, 8), (1, 7)),      # Conv1 → BN+ReLU
    ((1, 7), (1, 6)),      # BN+ReLU → Pool1
    ((1, 6), (1, 5)),      # Pool1 → Conv2
    ((1, 5), (1, 4)),      # Conv2 → BN+ReLU
    ((1, 4), (1, 3)),      # BN+ReLU → Pool2
    ((1, 3), (4, 9)),      # Pool2 → Conv3
    ((4, 9), (4, 8)),      # Conv3 → BN+ReLU
    ((4, 8), (4, 7)),      # BN+ReLU → Pool3
    ((4, 7), (4, 6)),      # Pool3 → Global Pool
    ((4, 6), (4, 5)),      # Global Pool → Dense
    ((4, 5), (4, 4)),      # Dense → Output
]

for (x1, y1), (x2, y2) in connections:
    ax.arrow(x1, y1 - 0.4, x2 - x1, y2 - y1 + 0.8, 
            head_width=0.15, head_length=0.1, fc='black', ec='black', linewidth=1.5)

# Title and formatting
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(2.5, 10)
ax.axis('off')
ax.set_aspect('equal')

plt.title('Spherical CNN Architecture', fontsize=16, weight='bold', pad=20)

# Add legend
info_text = """Data Flow:
• Input resolution decreases via HEALPix pooling
• Number of features (channels) increases
• Global pooling reduces spatial dims before classification"""
ax.text(3.5, 2.8, info_text, fontsize=9, family='monospace', 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.savefig('spherical_cnn_architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved spherical_cnn_architecture.png")

## Method 2: 3D Spherical Visualization

Visualize how the network processes data at different resolutions on the sphere.
Shows receptive fields growing with each layer.

In [ ]:
# Visualize network processing across different HEALPix resolutions
nsides = [256, 128, 64, 32]  # Matching the pooling hierarchy
num_filters = [3, 32, 64, 128]  # Channels at each layer

fig = plt.figure(figsize=(18, 12))

for idx, (nside, num_ch) in enumerate(zip(nsides, num_filters)):
    ax = fig.add_subplot(2, 2, idx + 1, projection='3d')
    
    NPIX = hp.nside2npix(nside)
    
    # Simulate channel importance (example: decreasing variance per channel)
    # In reality, you'd visualize learned feature importance
    channel_importance = np.exp(-np.arange(num_ch) / 4)  # Exponential decay
    
    # Create a fake feature map: random per pixel
    feature_map = np.random.randn(NPIX, num_ch).mean(axis=1)  # Average across channels
    feature_map = (feature_map - feature_map.min()) / (feature_map.max() - feature_map.min())
    
    # Colormap
    cmap = plt.cm.get_cmap('plasma')
    norm = plt.Normalize(vmin=0, vmax=1)
    
    # Sample pixels to avoid overcrowding
    step = max(1, NPIX // 500)
    pixels = np.arange(0, NPIX, step)
    
    # Plot each pixel
    for pix in pixels:
        bnd = hp.boundaries(nside, pix, nest=True)
        
        x, y, z = bnd[0], bnd[1], bnd[2]
        vertices = np.column_stack([x, y, z])
        
        # Color by feature activation
        color = cmap(norm(feature_map[pix]))
        
        poly = [[vertices[0], vertices[1], vertices[2], vertices[3]]]
        ax.add_collection3d(Poly3DCollection(poly, alpha=0.8, facecolor=color, 
                                             edgecolor='darkgray', linewidth=0.2))
    
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])
    ax.set_box_aspect([1, 1, 1])
    
    # Labels
    layer_name = ['Input', 'Conv Block 1', 'Conv Block 2', 'Conv Block 3'][idx]
    ax.set_title(f"{layer_name}\nnside={nside} | NPIX={NPIX} | {num_ch} channels", fontsize=12, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

plt.tight_layout()
plt.savefig('spherical_cnn_layers.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved spherical_cnn_layers.png")

## Visualizing Receptive Fields

Show how receptive fields grow on the sphere as you go deeper in the network.

In [ ]:
def get_pixel_neighbors(nside, pix, depth=1, nest=True):
    """
    Get all neighbors of a pixel up to a certain depth on the sphere.
    Simulates receptive field growth.
    """
    neighbors = {pix}
    to_visit = [pix]
    
    for _ in range(depth):
        new_neighbors = set()
        for pixel in to_visit:
            # Get immediate neighbors
            neighbor_list = hp.get_all_neighbours(nside, pixel, nest=nest)
            # Remove -1 (invalid pixels at poles)
            neighbor_list = neighbor_list[neighbor_list >= 0]
            new_neighbors.update(neighbor_list)
        
        neighbors.update(new_neighbors)
        to_visit = list(new_neighbors - neighbors)
    
    return np.array(list(neighbors))

# Visualize receptive field growth
nside = 64
center_pix = 0  # North pole
depths = [1, 2, 3]  # Receptive field depths

fig = plt.figure(figsize=(15, 5))

for idx, depth in enumerate(depths):
    ax = fig.add_subplot(1, 3, idx + 1, projection='3d')
    
    NPIX = hp.nside2npix(nside)
    
    # Get receptive field
    rf_pixels = get_pixel_neighbors(nside, center_pix, depth=depth)
    
    # Plot all pixels with receptive field highlighted
    for pix in range(NPIX):
        bnd = hp.boundaries(nside, pix, nest=True)
        
        x, y, z = bnd[0], bnd[1], bnd[2]
        vertices = np.column_stack([x, y, z])
        
        # Color: red for RF, light gray for others
        if pix in rf_pixels:
            color = 'red'
            alpha = 0.9
            edge_color = 'darkred'
            edge_width = 0.5
        else:
            color = 'lightgray'
            alpha = 0.3
            edge_color = 'gray'
            edge_width = 0.1
        
        poly = [[vertices[0], vertices[1], vertices[2], vertices[3]]]
        ax.add_collection3d(Poly3DCollection(poly, alpha=alpha, facecolor=color,
                                             edgecolor=edge_color, linewidth=edge_width))
    
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])
    ax.set_box_aspect([1, 1, 1])
    
    ax.set_title(f"Receptive Field (depth={depth})\n{len(rf_pixels)} pixels", fontsize=12, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

plt.tight_layout()
plt.savefig('spherical_receptive_fields.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved spherical_receptive_fields.png")

## Pooling Visualization

Show how HEALPix pooling reduces resolution while preserving spherical structure.

In [ ]:
# Visualize pooling: how pixels merge when downsampling
fig = plt.figure(figsize=(15, 5))

nsides_pool = [256, 128, 64]

for idx, nside in enumerate(nsides_pool):
    ax = fig.add_subplot(1, 3, idx + 1, projection='3d')
    
    NPIX = hp.nside2npix(nside)
    
    # Color by pixel index (normalized)
    cmap = plt.cm.get_cmap('tab20')
    
    # Sample to avoid overcrowding
    step = max(1, NPIX // 800)
    pixels = np.arange(0, NPIX, step)
    
    for pix_idx, pix in enumerate(pixels):
        bnd = hp.boundaries(nside, pix, nest=True)
        
        x, y, z = bnd[0], bnd[1], bnd[2]
        vertices = np.column_stack([x, y, z])
        
        # Cycle through colors
        color = cmap((pix_idx % 20) / 20)
        
        poly = [[vertices[0], vertices[1], vertices[2], vertices[3]]]
        ax.add_collection3d(Poly3DCollection(poly, alpha=0.85, facecolor=color,
                                             edgecolor='black', linewidth=0.3))
    
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])
    ax.set_box_aspect([1, 1, 1])
    
    ax.set_title(f"Resolution Level {idx}\nnside={nside} | NPIX={NPIX}", fontsize=12, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

plt.tight_layout()
plt.savefig('spherical_pooling.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved spherical_pooling.png")

## Summary

**Method 1 (Block Diagram):**
- Clean layer-by-layer flow
- Shows data shapes at each stage
- Good for papers and documentation

**Method 2 (3D Spherical):**
- Emphasizes spherical topology
- Visualizes feature activations
- Shows receptive field growth
- Illustrates pooling/resolution changes

**Use both together:** Block diagram for clarity, 3D for intuition.